# Generative Models and AI for Science[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week11_diffusion.ipynb)*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.***BME 6938 Advanced Biomedical AI, Week 11, reference notebook (ungraded)**This notebook follows the Week 11 lecture and its three clips. It builds a complete diffusion model from scratch, smallenough to train on a CPU in under a minute, on a synthetic two-dimensional stand-in for a protein backbone: a chain of16 points that is either helix-like or hairpin-like. Nothing here is real structural data, and nothing here is a proteindesign tool. The point is that every mechanism in RFdiffusion and in the AlphaFold 3 diffusion module is present in a formyou can read in full:| Section | Clip | What it builds ||---|---|---|| 1 | Clip 1, adding noise | synthetic backbone-like chains, the DDPM schedule, and the forward process || 2 | Clip 2, learning the way back | a small MLP denoiser, its training loop, and sampling from pure noise || 3 | Clip 3, steering the sampler | conditioning on a fixed motif, and classifier-free guidance || 4 | all three | what the toy does and does not tell you about the real systems |The figures in the text are from our own run on a desktop CPU, and yours should match unless a library has changed under it. Where the toy disagrees with the clip, the notebook says so: section 3.3 is one of those places.**Runtime.** About a minute on a desktop CPU, including training; not yet timed on Colab. No GPU, no downloads, no data files.

## 0. Setup

In [ ]:
import time                                                  # for timing the training loopfrom itertools import combinations                           # for listing all point pairsimport numpy as np                                           # arraysimport torch                                                 # tensors and autogradimport torch.nn as nn                                        # layersimport matplotlib.pyplot as plt                              # figuresfrom sklearn.linear_model import LogisticRegression          # a simple classifier used only to score our samplestorch.manual_seed(0)                                         # fixed seed, so the printed numbers are reproduciblerng = np.random.default_rng(0)                               # a separate generator for the synthetic datatorch.set_num_threads(2)                                     # keep CPU usage modest on a shared machineN = 16                                                       # points per chain, the same 16 C-alpha points as in clip 1print("torch", torch.__version__, "| running on CPU")

## 1. The forward process (clip 1, "Diffusion, part 1: adding noise")### 1.1 Synthetic backbone-like chainsA real protein backbone is a chain of C-alpha positions in three dimensions. We use two dimensions and 16 points so thatevery chain can be drawn on a page and every tensor is small. Two families stand in for secondary structure:- a **helix** seen from the side, which is a sine wave,- a **hairpin**, which is two antiparallel strands joined by a turn.Each chain gets a random amplitude or strand separation, a random rotation, and a little jitter, so the dataset is adistribution rather than two fixed shapes. This is synthetic data: no structures from the PDB, no patient data.

In [ ]:
def helix(r):    """A helix seen from the side: a sine wave with a random amplitude and number of turns."""    u = np.linspace(0, 1, N)                                 # 16 evenly spaced positions along the chain    a = r.uniform(0.35, 0.60)                                # amplitude of the wave, that is, the radius of the helix    k = r.uniform(2.0, 2.8)                                  # how many turns fit into the 16 residues    return np.stack([1.7 * (u - 0.5), a * np.sin(2 * np.pi * k * u)], 1)   # (16, 2) array of x, y positionsdef hairpin(r):    """Two antiparallel strands joined by a semicircular turn."""    u = np.linspace(0, 1, N)                                 # position along the chain, from 0 to 1    L = r.uniform(1.2, 1.7)                                  # half-length of each strand    s = r.uniform(0.35, 0.55)                                # half the distance between the two strands    x, y = np.empty(N), np.empty(N)                          # the coordinates we are about to fill in    for i, v in enumerate(u):        if v < 0.44:                                         # first strand: left to right along y = +s            x[i], y[i] = -L + (v / 0.44) * 2 * L, s        elif v < 0.56:                                       # the turn: a semicircle of radius s            a = (v - 0.44) / 0.12 * np.pi            x[i], y[i] = L + s * np.sin(a), s * np.cos(a)        else:                                                # second strand: right to left along y = -s            x[i], y[i] = L - ((v - 0.56) / 0.44) * 2 * L, -s    return np.stack([x, y], 1)def sample_chain(r, cls):    """One chain of the requested class, randomly rotated, centred, and jittered."""    p = helix(r) if cls == 0 else hairpin(r)                 # class 0 is a helix, class 1 is a hairpin    th = r.uniform(0, 2 * np.pi)                             # a random orientation, so the model cannot memorise one pose    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])   # the 2 by 2 rotation matrix    p = (p - p.mean(0)) @ R.T                                # centre the chain on the origin, then rotate it    return p + r.normal(0, 0.02, p.shape)                    # a little jitter, as real coordinates are never exactdef dataset(n, r):    """n chains with their class labels, half helices and half hairpins on average."""    y = r.integers(0, 2, n)                                  # one label per chain, 0 or 1    return np.stack([sample_chain(r, c) for c in y]), y      # shapes (n, 16, 2) and (n,)X_raw, y_train = dataset(4000, rng)                          # the training setSCALE = X_raw.std()                                          # one number: the overall spread of the coordinatesX_train = X_raw / SCALE                                      # rescale so the data has unit standard deviationprint("training set", X_train.shape, "| scale divided out:", round(float(SCALE), 3),      "| std after scaling:", round(float(X_train.std()), 3))

That rescaling matters more than it looks. The forward process ends at a standard Gaussian, so the data has to live atroughly the same scale as a standard Gaussian for the schedule to make sense. Dividing by one number is the whole of thenormalization here; in RFdiffusion the equivalent step is their choice of coordinate scaling for the translations.

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(13, 4.2))              # two rows: helices on top, hairpins belowfor row, cls in enumerate((0, 1)):    idx = np.where(y_train == cls)[0][:6]                    # the first six chains of this class    for col, i in enumerate(idx):        P = X_train[i]                                       # one chain, shape (16, 2)        ax[row, col].plot(P[:, 0], P[:, 1], "-o", ms=4, color="#0F766E")   # bonds and C-alpha points        ax[row, col].set_aspect("equal"); ax[row, col].axis("off")         # equal axes, no frame    ax[row, 0].set_title(["helix", "hairpin"][cls], loc="left", fontsize=11)fig.suptitle("Synthetic backbone-like chains (illustrative, not real structures)", y=1.0)plt.tight_layout(); plt.show()

### 1.2 The scheduleClip 1 gave the closed form for the forward process:    x_t = sqrt(signal_t) * x_0 + sqrt(1 - signal_t) * noisewith `signal_t` the cumulative product of `1 - beta` up to step `t`. The standard DDPM schedule takes `beta` linearly from0.0001 to 0.02 over 1,000 steps. The next cell recomputes the four checkpoints shown in the clip rather than quoting them.

In [ ]:
STEPS = 1000                                                 # total number of forward stepsbetas = torch.linspace(1e-4, 0.02, STEPS)                    # the variance added at each step, rising linearlyalphas = 1.0 - betas                                         # the fraction of the signal each step keepsabar = torch.cumprod(alphas, 0)                              # signal_t: the product of all the alphas up to step tprint(f"{'t':>5}  {'signal weight':>14}  {'noise weight':>13}")for t in (1, 100, 250, 500, 999):                            # the checkpoints shown on screen in clip 1    s = float(abar[t - 1].sqrt())                            # sqrt(signal_t), the weight on the real structure    q = float((1 - abar[t - 1]).sqrt())                      # sqrt(1 - signal_t), the weight on the noise    print(f"{t:>5}  {s:>14.3f}  {q:>13.3f}")

### 1.3 Noising one chainThe forward process needs no model and no training: one line of arithmetic takes a clean chain to any step `t`. Using**one** fixed noise draw for every `t` makes the picture a smooth interpolation from structure to noise, which is exactlywhat the clip animates.

In [ ]:
def q_sample(x0, t, noise):    """The forward process: mix a clean chain x0 with noise, using the weights for step t."""    a = abar[t - 1]                                          # signal_t for this step    return a.sqrt() * x0 + (1 - a).sqrt() * noise            # the closed form from clip 1x0 = torch.tensor(X_train[np.where(y_train == 0)[0][0]], dtype=torch.float32)   # one helixeps = torch.randn(x0.shape, generator=torch.Generator().manual_seed(4))        # one fixed noise draw, reused belowfig, ax = plt.subplots(1, 6, figsize=(13, 2.6))for k, t in enumerate((1, 100, 250, 500, 800, 999)):         # the chain at six points along the forward process    P = q_sample(x0, t, eps).numpy()                         # the noised chain at step t    col = "#0F766E" if t <= 250 else "#6B7280"               # teal while structure is visible, grey once it is gone    ax[k].plot(P[:, 0], P[:, 1], "-o", ms=3.5, color=col)    ax[k].set_title(f"t = {t}\nsignal {float(abar[t-1].sqrt()):.2f}", fontsize=10)    ax[k].set_xlim(-3, 3); ax[k].set_ylim(-3, 3)             # the same axes everywhere, so the collapse is visible    ax[k].set_aspect("equal"); ax[k].axis("off")plt.tight_layout(); plt.show()

By `t = 999` the chain is a Gaussian cloud, and nothing in it says which chain it started from. Every training example endsin the same distribution, which is why sampling can start from noise alone.## 2. The reverse process (clip 2, "Diffusion, part 2: learning the way back")### 2.1 The denoiserThe network sees the noisy chain `x_t` and the step number `t`, and predicts the noise that was added. It is a plain MLPover the flattened 32 coordinates. Two extra inputs are concatenated:- a **time embedding**: sines and cosines of `t` at several frequencies, so the network can tell step 50 from step 950,- a **class input**: helix, hairpin, or "unconditional". During training the label is replaced by "unconditional" 15% of  the time, which is exactly what classifier-free guidance in section 3.2 needs.This is the role RoseTTAFold plays inside RFdiffusion, with about five orders of magnitude fewer parameters.

In [ ]:
class Denoiser(nn.Module):    """Predicts the noise that was added to a chain, given the chain, the step number, and a class."""    def __init__(self, d=2 * N, h=256):        super().__init__()        self.freqs = torch.exp(torch.linspace(0, np.log(1000), 8))          # 8 frequencies for the time embedding        self.net = nn.Sequential(                                           # d coordinates + 16 time features + 3 class flags            nn.Linear(d + 16 + 3, h), nn.SiLU(),            nn.Linear(h, h), nn.SiLU(),            nn.Linear(h, h), nn.SiLU(),            nn.Linear(h, d))                                                # output: one predicted noise value per coordinate    def time_embedding(self, t):        """Sines and cosines of the step number at 8 frequencies, giving 16 numbers."""        a = t[:, None].float() / STEPS * self.freqs[None, :]                # scale t into [0, 1], then spread over frequencies        return torch.cat([a.sin(), a.cos()], 1)                             # (batch, 16)    def forward(self, x, t, c):        """x: (batch, 32) flattened chains. t: (batch,) step numbers. c: (batch,) class, 0 helix, 1 hairpin, 2 none."""        onehot = torch.nn.functional.one_hot(c, 3).float()                  # (batch, 3)        return self.net(torch.cat([x, self.time_embedding(t), onehot], 1))  # (batch, 32) predicted noisemodel = Denoiser()print("parameters:", sum(p.numel() for p in model.parameters()))

### 1.4 Watch the forward process run

Section 1.3 printed four checkpoints. The animation runs the same closed form continuously, on one synthetic backbone, so
you can see where the structure actually goes.

The left panel is the chain at step `t`, drawn against the clean chain in grey. The right panel is `signal_t`, the
fraction of the original that survives, with a marker at the current step.

The thing worth watching is how uneven the loss of structure is. Almost nothing happens for the first hundred steps, the
chain is unrecognisable somewhere in the middle, and the last few hundred steps only rearrange noise. That is why the
sampler in section 2.3 spends most of its effort in the middle of the schedule, and why the model is trained on steps
drawn uniformly: the interesting ones have to be hit often enough.

This is the forward direction only, which needs no model at all. The learned reverse is section 2.3.

In [ ]:
# An animation, drawn by the code below. It renders with to_jshtml, which needs no
# ffmpeg and plays in Colab, in Jupyter, and on nbviewer. Press play, or drag the
# slider to step through it frame by frame.
import numpy as np                                           # arrays
import matplotlib.pyplot as plt                              # figures
from matplotlib.animation import FuncAnimation               # the animation itself
from IPython.display import HTML                             # shows it in the notebook

UF, ORANGE, GREY = "#0021A5", "#FA4616", "#6B7280"           # the course colours
plt.rcParams["figure.dpi"] = 78                              # keeps the rendered animation light

N, STEPS = 16, 1000                                          # points in a chain, steps in the schedule
rng = np.random.default_rng(0)

# One synthetic helix, the same family of shapes as section 1.1.
u = np.linspace(0, 1, N)
chain = np.stack([u * 2 - 1, 0.5 * np.sin(2.4 * 2 * np.pi * u)], 1)
chain = chain / np.abs(chain).max()                          # the rescaling section 1.1 explains

betas = np.linspace(1e-4, 0.02, STEPS)                       # the standard DDPM schedule
abar = np.cumprod(1.0 - betas)                               # signal_t: how much of x_0 is left at step t
eps = rng.normal(0, 1, chain.shape)                          # one fixed noise draw, so the animation is smooth

TS = np.unique(np.linspace(0, STEPS - 1, 45).astype(int))    # the steps we render

fig, (ax, sched) = plt.subplots(1, 2, figsize=(9.2, 3.6),
                                gridspec_kw={"width_ratios": [1.15, 1]})
fig.subplots_adjust(wspace=0.3)

def draw(f):
    t = TS[f]
    a = abar[t]
    x = np.sqrt(a) * chain + np.sqrt(1 - a) * eps            # the closed form, straight from clip 1
    ax.clear(); sched.clear()

    ax.plot(chain[:, 0], chain[:, 1], "-o", ms=3.5, color="#D1D5DB", lw=1.2)   # the clean chain, for reference
    ax.plot(x[:, 0], x[:, 1], "-o", ms=4.5, color=UF, lw=1.6)
    ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"t = {t}      signal_t = {a:.3f}", loc="left", fontsize=11, color="#374151")

    sched.plot(np.arange(STEPS), abar, color="#C7CDD1", lw=1.6)
    sched.plot(np.arange(t + 1), abar[:t + 1], color=UF, lw=2.2)
    sched.scatter([t], [a], color=ORANGE, s=38, zorder=3)
    sched.set_xlim(0, STEPS); sched.set_ylim(0, 1.02)
    sched.set_xlabel("step t", fontsize=10)
    sched.set_ylabel("signal_t", fontsize=10)
    sched.spines[["top", "right"]].set_visible(False)
    return []

ani = FuncAnimation(fig, draw, frames=len(TS), interval=110)
plt.close(fig)
HTML(ani.to_jshtml(fps=9))

### 2.2 TrainingThe training objective is one line: take a clean chain, pick a random step `t`, noise the chain to that step, and ask thenetwork to recover the noise. The loss is the mean squared error between the real noise and the predicted noise. Nothingin this loop knows anything about proteins.

In [ ]:
X = torch.tensor(X_train, dtype=torch.float32).reshape(-1, 2 * N)   # (4000, 32), one flattened chain per rowY = torch.tensor(y_train)                                           # (4000,) class labelsopt = torch.optim.Adam(model.parameters(), 2e-3)                    # Adam at a fairly high learning rate; the model is tinyTRAIN_STEPS, BATCH = 20000, 128t0 = time.time()for i in range(TRAIN_STEPS):    j = torch.randint(0, len(X), (BATCH,))                          # a random minibatch of chains    x0 = X[j]                                                       # the clean chains    c = Y[j].clone()                                                # their class labels    c[torch.rand(BATCH) < 0.15] = 2                                 # label dropout: 15% are trained unconditionally    t = torch.randint(1, STEPS + 1, (BATCH,))                       # a random step for each chain in the batch    a = abar[t - 1][:, None]                                        # signal_t, one per chain, shaped for broadcasting    noise = torch.randn_like(x0)                                    # the noise we will ask the model to recover    xt = a.sqrt() * x0 + (1 - a).sqrt() * noise                     # the forward process, as in section 1.3    loss = ((model(xt, t, c) - noise) ** 2).mean()                  # mean squared error on the predicted noise    opt.zero_grad(); loss.backward(); opt.step()    if i % 4000 == 0 or i == TRAIN_STEPS - 1:        print(f"step {i:>6}  loss {float(loss.detach()):.4f}  ({time.time() - t0:.0f} s)")print(f"trained in {time.time() - t0:.0f} s on CPU")

### 2.3 SamplingSampling runs the reverse process: start from pure noise and step from `t = 1000` down to `t = 1`. At each step the modelpredicts the noise, which gives an estimate of the clean chain, and the sampler moves a little way toward it and adds alittle fresh noise. The `keep` argument is the conditioning from section 3.1; ignore it for now.

In [ ]:
@torch.no_grad()                                                    # no gradients: this is inference onlydef sample(n, cls=2, w=1.0, keep=None, x0ref=None, seed=0):    """Generate n chains. cls: 0 helix, 1 hairpin, 2 unconditional. w: guidance strength (section 3.2).       keep: indices of points to hold fixed. x0ref: the chains whose kept points are imposed."""    g = torch.Generator().manual_seed(seed)                         # a generator, so every call is reproducible    x = torch.randn(n, 2 * N, generator=g)                          # start from pure noise, as at t = 999 in clip 1    for t in range(STEPS, 0, -1):                                   # walk the schedule backwards        tt = torch.full((n,), t)                                    # the same step number for the whole batch        a, ap = abar[t - 1], (abar[t - 2] if t > 1 else torch.tensor(1.0))   # signal_t and signal_(t-1)        e_uncond = model(x, tt, torch.full((n,), 2))                # the unconditional noise prediction        if cls == 2:            e = e_uncond                                            # no class requested: use it directly        else:            e_cond = model(x, tt, torch.full((n,), cls))            # the noise prediction for the requested class            e = e_uncond + w * (e_cond - e_uncond)                  # the guidance formula from clip 3        x0_hat = ((x - (1 - a).sqrt() * e) / a.sqrt()).clamp(-3, 3) # the implied clean chain, clipped for stability        mean = (ap.sqrt() * (1 - alphas[t - 1]) / (1 - a)) * x0_hat \             + (alphas[t - 1].sqrt() * (1 - ap) / (1 - a)) * x      # the DDPM posterior mean: a step toward x0_hat        if t > 1:            var = (1 - ap) / (1 - a) * betas[t - 1]                 # the posterior variance            x = mean + var.sqrt() * torch.randn(x.shape, generator=g)   # the step, plus a little fresh noise        else:            x = mean                                                # the last step adds no noise        if keep is not None and t > 1:                              # conditioning, explained in section 3.1            an = abar[t - 2]                                        # noise the known points to the step we just reached            known = an.sqrt() * x0ref + (1 - an).sqrt() * torch.randn(x0ref.shape, generator=g)            x = x.reshape(n, N, 2)                                  # unflatten so we can index individual points            x[:, keep, :] = known.reshape(n, N, 2)[:, keep, :]      # overwrite the kept points with the known ones            x = x.reshape(n, 2 * N)                                 # flatten again for the next pass through the model    return x.reshape(n, N, 2)                                       # (n, 16, 2) generated chainst0 = time.time()gen_helix = sample(8, cls=0, w=1.0, seed=5).numpy()                 # eight chains asked to be helicesgen_hairpin = sample(8, cls=1, w=1.0, seed=5).numpy()               # eight chains asked to be hairpinsprint(f"sampled 16 chains in {time.time() - t0:.1f} s")

In [ ]:
real_of = {c: X_train[y_train == c] for c in (0, 1)}                # the real chains of each class, for comparisonfig, ax = plt.subplots(4, 8, figsize=(14, 7))rows = [(real_of[0][:8], "real helix"), (gen_helix, "generated helix"),        (real_of[1][:8], "real hairpin"), (gen_hairpin, "generated hairpin")]for r, (chains, label) in enumerate(rows):    for c in range(8):        P = chains[c]        ax[r, c].plot(P[:, 0], P[:, 1], "-o", ms=3.5,                      color=("#6B7280" if "real" in label else "#0021A5"))   # grey for real, UF blue for generated        ax[r, c].set_aspect("equal"); ax[r, c].axis("off")    ax[r, 0].set_title(label, loc="left", fontsize=11)plt.tight_layout(); plt.show()

Look at the figure before reading any numbers. Our run gave recognizable helices and hairpins: the class is never indoubt. The flaws are the ones you would expect from a 153,000-parameter MLP trained for half a minute. The generatedhairpin strands are slightly wavier than the real ones, which are perfectly straight, and the generated helices vary morein amplitude and pitch along a single chain than any real helix does.Two measurements make that more precise. The first is the **bond length**, the distance between consecutive points: a realchain has a characteristic spacing, and a model that has not learned local geometry gets it wrong. The second is a**classifier**, a logistic regression on the 120 pairwise distances within a chain (which is invariant to rotation andtranslation, unlike the raw coordinates). It reaches 100% on held-out real chains, so it is a fair judge of whether agenerated chain belongs to the class we asked for.

In [ ]:
PAIRS = list(combinations(range(N), 2))                             # all 120 point pairs in a 16-point chaindef features(P):    """The 120 pairwise distances within each chain: a description that does not change if the chain is rotated."""    P = np.asarray(P)    return np.stack([np.linalg.norm(P[:, i] - P[:, j], axis=1) for i, j in PAIRS], 1)def bonds(P):    """The 15 distances between consecutive points of each chain."""    return np.linalg.norm(np.diff(np.asarray(P), axis=1), axis=2)clf = LogisticRegression(max_iter=2000).fit(features(X_train), y_train)     # train the judge on real chainsX_test, y_test = dataset(1000, np.random.default_rng(7))                   # fresh real chains it has never seenprint("classifier accuracy on held-out real chains:", round(clf.score(features(X_test / SCALE), y_test), 3))print()print(f"{'set':>20}  {'bond length mean':>17}  {'class match':>12}")for chains, label, cls in ((real_of[0][:8], "real helix", 0), (gen_helix, "generated helix", 0),                           (real_of[1][:8], "real hairpin", 1), (gen_hairpin, "generated hairpin", 1)):    match = float((clf.predict(features(chains)) == cls).mean())           # fraction judged to be the requested class    print(f"{label:>20}  {bonds(chains).mean():>17.3f}  {match:>12.2f}")

## 3. Steering the sampler (clip 3, "Diffusion, part 3: steering the sampler")### 3.1 Conditioning on a motifClip 3 made the distinction that matters for protein design: **conditioning** gives the denoiser information it mustrespect, while **guidance** reshapes the prediction after the fact. RFdiffusion conditions; it does not useclassifier-free guidance.The toy version of motif scaffolding is the simplest kind of conditioning there is. Pick four consecutive points and callthem the motif. At every reverse step, overwrite those four coordinates with the known motif noised to the current step.The sampler is then free only in the other twelve points, so it has to build a scaffold that connects to a motif it doesnot control. Note what this does **not** do: it never tells the network about the motif, so the network has to discoverthe constraint through the coordinates it sees. Real RFdiffusion feeds the motif in as an input.

In [ ]:
MOTIF = list(range(6, 10))                                          # four consecutive points held fixedn_cond = 48                                                         # how many conditioned chains to generateref = torch.tensor(X_train[:n_cond], dtype=torch.float32).reshape(n_cond, 2 * N)   # the chains the motifs come fromcond = sample(n_cond, cls=2, keep=MOTIF, x0ref=ref, seed=2).numpy() # scaffold around each motif, class unspecifiedref3 = ref.numpy().reshape(n_cond, N, 2)                            # the reference chains, reshaped back to (n, 16, 2)err = np.linalg.norm(cond[:, MOTIF, :] - ref3[:, MOTIF, :], axis=2) # distance between each generated and known motif pointrmse = float(np.sqrt((err ** 2).mean()))                            # one number for how well the motif was heldprint("motif RMSE:", round(rmse, 4), "model units =", round(rmse * SCALE, 4), "display units")print("mean |coordinate| of a chain:", round(float(np.abs(X_train).mean()), 3), "model units")print()free = sample(n_cond, cls=2, seed=3).numpy()                        # unconditioned chains, for comparisonprint("bond length mean, conditioned:", round(bonds(cond).mean(), 3),      "| unconditioned:", round(bonds(free).mean(), 3),      "| real:", round(bonds(X_train).mean(), 3))

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(13, 4.6))for c in range(6):    for r, P in enumerate((ref3[c], cond[c])):                              # top: the original chain; bottom: the scaffold        ax[r, c].plot(P[:, 0], P[:, 1], "-o", ms=4, color="#0F766E")        # the whole chain in teal        ax[r, c].plot(P[MOTIF, 0], P[MOTIF, 1], "o", ms=8, color="#F59E0B") # the motif points in amber, as in clip 3        ax[r, c].set_aspect("equal"); ax[r, c].axis("off")    both = np.concatenate([ref3[c], cond[c]])                               # the same axes for both rows of a column    for r in (0, 1):        ax[r, c].set_xlim(both[:, 0].min() - 0.3, both[:, 0].max() + 0.3)        ax[r, c].set_ylim(both[:, 1].min() - 0.3, both[:, 1].max() + 0.3)ax[0, 0].set_title("original", loc="left", fontsize=11)                 # row labels, kept short so they fit one panelax[1, 0].set_title("scaffold", loc="left", fontsize=11)plt.tight_layout(); plt.show()

The motif RMSE is a small fraction of the chain's own scale, so the motif really is held in place. But look at the figurebefore concluding anything: several of the scaffolds are nearly identical to the chain the motif was taken from. That isnot what motif scaffolding is supposed to do, so it is worth measuring rather than assuming.The question is whether one motif gives many scaffolds. The next cell answers it directly: take a single motif, generatetwelve scaffolds for it, and compare how much they differ from each other with how much twelve unconditioned samplesdiffer.

In [ ]:
one = ref[:1].repeat(12, 1)                                         # the same reference chain twelve timesmany = sample(12, cls=2, keep=MOTIF, x0ref=one, seed=11).numpy()    # twelve scaffolds around one motiffree12 = sample(12, cls=2, seed=12).numpy()                         # twelve unconditioned chains, for comparisontail = [i for i in range(N) if i not in MOTIF]                      # the twelve points the sampler is free to choosedef spread(chains):    """Mean distance between pairs of chains, over the free points only, in raw coordinates."""    F = np.asarray(chains)[:, tail, :].reshape(len(chains), -1)    D = np.linalg.norm(F[:, None, :] - F[None, :, :], axis=2)    return float(D[np.triu_indices(len(F), 1)].mean())ref_rep = ref3[0]                                                   # the chain the motif came fromto_ref = np.linalg.norm(many[:, tail, :] - ref_rep[tail, :], axis=2).mean()   # how close each scaffold is to itprint("spread among 12 scaffolds of one motif:", round(spread(many), 3))print("spread among 12 unconditioned chains: ", round(spread(free12), 3))print("mean distance from a scaffold to the original chain:", round(float(to_ref), 3))

In [ ]:
fig, ax = plt.subplots(1, 7, figsize=(14, 2.5))for c, (P, ttl) in enumerate([(ref_rep, "original")] + [(many[i], f"scaffold {i+1}") for i in range(6)]):    ax[c].plot(P[:, 0], P[:, 1], "-o", ms=4, color=("#6B7280" if c == 0 else "#0021A5"))    ax[c].plot(P[MOTIF, 0], P[MOTIF, 1], "o", ms=8, color="#F59E0B")        # the motif, identical in all seven panels    ax[c].set_title(ttl, fontsize=10); ax[c].set_aspect("equal"); ax[c].axis("off")    ax[c].set_xlim(-2.6, 2.6); ax[c].set_ylim(-2.6, 2.6)plt.tight_layout(); plt.show()

In our run the twelve scaffolds for one motif were far less varied than twelve free samples, and each one stayed close tothe chain the motif came from. The honest conclusion is that in this toy a four-point motif nearly determines the rest ofthe chain: there are only two shape families and sixteen points, so fixing a quarter of them leaves little freedom. Realmotif scaffolding fixes a handful of residues in a protein of hundreds, where the scaffold is genuinely underdetermined.If you want the toy to behave more like the real problem, shrink `MOTIF` to two points and rerun. That is exercise 2.This is still the toy version of the result in clip 3, where RFdiffusion solved 23 of 25 motif-scaffolding benchmarks,but the resemblance is in the mechanism, not in the difficulty.### 3.2 Guidance strength against diversityClassifier-free guidance combines two predictions from the same network:    e = e_uncond + w * (e_cond - e_uncond)with `w = 0` giving the unconditional model, `w = 1` giving exactly the conditional model, and `w > 1` extrapolatingpast it. Clip 3 showed the arithmetic with 0.2, 0.5, and `w = 3`, which gives 1.1.We sweep `w` and measure two things on 48 generated hairpins:- **realism**: the mean distance from each sample to the nearest *real* hairpin, in the rotation-invariant feature space.  Lower is better.- **diversity**: the mean distance between pairs of samples. Higher means the 48 samples differ more from each other.The textbook claim is that raising `w` buys fidelity and costs diversity. Watch whether that is what happens here.

In [ ]:
feat_real = {c: features(real_of[c]) for c in (0, 1)}               # the real chains of each class, as featuresdef realism(chains, cls):    """Mean distance from each generated chain to the nearest real chain of the requested class. Lower is better."""    F = features(chains)    D = np.linalg.norm(F[:, None, :] - feat_real[cls][None, :, :], axis=2)   # (n_generated, n_real) distances    return float(D.min(1).mean())def diversity(chains):    """Mean distance between pairs of chains in the set. Higher means the samples differ more from each other."""    F = features(chains)    D = np.linalg.norm(F[:, None, :] - F[None, :, :], axis=2)    return float(D[np.triu_indices(len(F), 1)].mean())print(f"{'w':>5}  {'class match':>12}  {'realism':>8}  {'diversity':>10}  {'bond mean':>10}")ws, rs, ds = [], [], []for w in (0.0, 1.0, 2.0, 3.0, 5.0, 8.0):                            # w = 0 is unconditional, w = 1 is conditional    g = sample(48, cls=1, w=w, seed=1).numpy()                      # 48 chains asked to be hairpins    match = float((clf.predict(features(g)) == 1).mean())    ws.append(w); rs.append(realism(g, 1)); ds.append(diversity(g))    print(f"{w:>5.1f}  {match:>12.2f}  {rs[-1]:>8.2f}  {ds[-1]:>10.2f}  {bonds(g).mean():>10.3f}")print()print(f"{'real hairpins':>5}  {'':>12}  {realism(real_of[1][:48], 1):>8.2f}  {diversity(real_of[1][:48]):>10.2f}"      f"  {bonds(real_of[1]).mean():>10.3f}")

In [ ]:
k = [i for i, w in enumerate(ws) if w >= 1]                         # plot w >= 1 only: the w = 0 point is off the scalefig, ax = plt.subplots(1, 2, figsize=(11, 3.6))ax[0].plot([ws[i] for i in k], [rs[i] for i in k], "-o", color="#0021A5")ax[0].set_ylabel("distance to nearest real hairpin")ax[0].set_title(f"realism, lower is better (w = 0 is off scale at {rs[0]:.1f})", fontsize=11)ax[1].plot([ws[i] for i in k], [ds[i] for i in k], "-o", color="#0F766E")ax[1].set_ylabel("mean distance between samples")ax[1].set_title(f"diversity (w = 0 is off scale at {ds[0]:.1f})", fontsize=11)for a in ax:    a.set_xlabel("guidance strength w"); a.grid(alpha=0.3)plt.tight_layout(); plt.show()

### 3.3 What the toy does and does not reproduceThree things came out of that sweep, and only the first two are what the clip led you to expect.1. **Conditioning works, and `w = 1` is where it starts.** At `w = 0` the sampler ignores the request: it matched the   requested class about half the time, which is chance, and both its realism and its diversity numbers reflect a set of   samples drawn from both classes. At `w = 1` the class match goes to 1.00 and realism improves sharply.2. **`w = 0` and `w = 1` are not arbitrary points on a scale.** `w = 1` reduces the guidance formula to the conditional   prediction exactly, so it is the conditional model, not a guided one. Everything above 1 is extrapolation.3. **Raising `w` past about 2 made our samples *worse*, not sharper, and did not monotonically reduce diversity.**   Realism was best near `w = 2` and got steadily worse from there; diversity fell from `w = 1` to `w = 2` and then rose   again by `w = 8`. The rise is not a return of useful variety: it is the samples distorting in different directions.   The textbook trade-off that clip 3 states is the rule of thumb from large image diffusion models, and this toy does   not reproduce it cleanly. Reporting that is the point of running the experiment rather than asserting the result.If you want to push on this, the most informative single change is to train longer. A denoiser that has not learned thedata well has an unconditional prediction that is itself poor, and extrapolating away from a poor prediction is whatdegrades the samples at large `w`.## 4. From this toy to RFdiffusion and AlphaFold 3Everything structural in this notebook is in the real systems, and everything that makes the real systems work is missinghere. Both halves are worth being precise about.**The same.** The forward process is the same arithmetic with the same kind of schedule. The training objective is thesame: predict the noise, which is the same thing as predicting the clean structure. The reverse process is the same loop.Conditioning by fixing coordinates is the same idea as motif scaffolding. The guidance formula is the one from clip 3.**Different, and it is the difference that matters.**| This notebook | RFdiffusion | AlphaFold 3 ||---|---|---|| 2D points, no chemistry | residue frames: a C-alpha position plus an N-C-alpha-C orientation | raw atom coordinates, no frames and no torsion angles || a 153,000-parameter MLP trained from scratch in 30 seconds | fine-tuned from pretrained RoseTTAFold weights | a 48-block pairformer feeding a diffusion module || 1,000 sampling steps | typically 200 steps, often with no fresh noise added (17 of the 23 solved benchmarks) | a diffusion module over atoms || no self-conditioning | the previous x0 prediction is fed back as a template | its own equivalent || a chain of points is the final output | ProteinMPNN designs about 8 sequences per backbone | proteins, nucleic acids, ligands, ions in one model || scored by a logistic regression we trained ourselves | scored in silico by AlphaFold-family predictions, then in a wet lab: 19% experimental success across five targets | scored against experimental structures |The last row is the one to carry into Thursday's paper session. Our classifier and our diffusion model were trained on thesame 4,000 synthetic chains, so of course they agree. The same critique applies, with much higher stakes, when designs arefiltered by a structure predictor trained on the same PDB that trained the designer. The number in the RFdiffusion paperthat is immune to that objection is the 19%, and the 28 nM influenza binder whose cryo-EM structure matched the design to0.63 angstroms, because those came from a laboratory.**Things to try**1. Raise `TRAIN_STEPS` to 60,000 and rerun section 3.2. Does the damage at large `w` get smaller?2. Move `MOTIF` to the two ends of the chain (`[0, 1, 14, 15]`) instead of the middle. Scaffolding between two fixed   anchors is harder; does the motif RMSE hold up?3. Cut sampling from 1,000 steps to 100 by striding the loop. RFdiffusion uses 200. How much quality do you lose?4. Remove the fresh noise from the sampler (set `var` to zero). RFdiffusion found this helped. Does it help here, and what   happens to diversity?